In [251]:
import xml.etree.ElementTree as ET
import numpy as np
import meshio as mio
import json
import os

import meshplot as mp

In [252]:
def gmsh_dim_tags(points, cells, tags):
    dim_tags = np.tile([3, 1], [points.shape[0], 1])
    for i, cell in enumerate(cells):
        for vi in cell:
            dim_tags[vi, 1] = tags[i]
    return dim_tags


def gmsh_cell_tags(tags):
    cell_tags = [[tags[0]]]
    for tag in tags[1:]:
        if tag == cell_tags[-1][-1]:
            cell_tags[-1].append(tag)
        else:
            cell_tags.append([tag])
    return cell_tags


def split_cells_by_tags(tags, cells):
    assert(len(tags) == len(cells))
    split_cells = [[cells[0]]]
    for i, tag in enumerate(tags[1:]):
        if tag == tags[i]:
            split_cells[-1].append(cells[i + 1])
        else:
            split_cells.append([cells[i + 1]])
    return split_cells


def save_msh(file, points, tets, tags):
    dim_tags = gmsh_dim_tags(points, tets, tags)
    cell_tags = gmsh_cell_tags(tags)

    tmp = [("tetra", np.array(cells)) for cells in split_cells_by_tags(tags, tets)]
    # print([("tetra", T)])
    # [("tetra", T)]


    mesh = mio.Mesh(
        points,
        tmp,
        point_data={"gmsh:dim_tags": dim_tags},
        cell_data={"gmsh:physical": cell_tags, "gmsh:geometrical": cell_tags}
    )

    mio.write(file, mesh, binary=True, file_format="gmsh")

In [253]:
def load_control(control, args):
    if control is None:
        return
        
    tsn = control.find("time_steps")
    ssn = control.find("step_size")
    an = control.find("analysis")

    
    if tsn is not None and ssn is not None and an is not None:
        if an.attrib["type"] == "dynamic":
            time_steps = int(tsn.text)
            step_size = float(ssn.text)

            args["time"] = {
                "tend": step_size * time_steps,
                "time_steps": time_steps
            }

In [254]:
def load_materials(febio):
    materials = {}

    material_parent = febio.find("Material")
    
    for material_node in material_parent.iter("material"):
        material = material_node.attrib["type"]
        mid = int(material_node.attrib["id"])+1
        
        E = float(material_node.find("E").text)
        nu = float(material_node.find("v").text)
                  
        if material_node.find("density") is None:
            rho = 1
        else:
            rho = float(material_node.find("density").text)

        mat = ""
        if material == "neo-Hookean":
            mat = "NeoHookean"
        elif material == "isotropic elastic":
            mat = "LinearElasticity"
        else:
            print("Unsupported material {}, reverting to isotropic elastic".format(material))
            mat = "LinearElasticity"
            

        materials[mid] = {"id": mid, "E": E, "nu": nu, "rho": rho, "type": mat}
        
    return materials

In [255]:
def load_nodes(geometry):
    vertices = []
    for nodes in geometry.iter("Nodes"):
        for child in nodes.iter("node"):
            pos_str = child.text
            vs = pos_str.split(",")
            assert(len(vs) == 3);
            
            vertices.append([float(vs[0]), float(vs[1]), float(vs[2])])


    V = np.array(vertices)
    
    return V

In [256]:
def load_elements(geometry, numV, materials):
    els = []
    nodes = []
    mids = []
    
    order = 1
    
    is_hex = False
    types = ""
    
    for elements in geometry.iter("Elements"):
        el_type = elements.attrib["type"]
        mid = int(elements.attrib["mat"])+1

        if el_type != "tet4" and el_type != "tet10" and el_type != "tet20" and el_type != "hex8":
            print("Unsupported elemet type {}".format(el_type))
            continue


        if len(types) == 0:
            if el_type.startswith("tet"):
                types = "tet"
            else:
                types = "hex"
        elif not el_type.startswith(types, 0):
            print("Unsupported elemet type {} since the mesh contains also {}".format(el_type, types))
            continue

        if el_type == "tet4":
            order = max(1, order)
        elif el_type == "tet10":
            order = max(2, order)
        elif el_type == "tet20":
            order = max(3, order)
        elif el_type == "hex8":
            order = max(1, order)
            is_hex = True
                

        for child in elements.iter("elem"):
                ids = child.text
                tt = ids.split(",");
                assert(len(tt) >= 4);
                
                node_size = 8 if is_hex else 4

                els.append([])
                
                for n in range(node_size):
                    els[-1].append(int(tt[n]) - 1)
                    assert(els[-1][n] < numV)

                nodes.append([])
                mids.append(mid)
                
                for n in range(len(tt)):
                    nodes[-1].append(int(tt[n]) - 1)

                if el_type == "tet10":
                    assert(len(nodes[-1]) == 10)
                    nodes[-1][8], nodes[-1][9] = nodes[-1][9], nodes[-1][8]
                elif el_type == "tet20":
                    assert(len(nodes[-1]) == 20)
                    nodes[-1][8], nodes[-1][9] = nodes[-1][9], nodes[-1][8]
                    nodes[-1][10], nodes[-1][11] = nodes[-1][11], nodes[-1][10]
                    nodes[-1][12], nodes[-1][15] = nodes[-1][15], nodes[-1][12]
                    nodes[-1][13], nodes[-1][14] = nodes[-1][14], nodes[-1][13]
                    nodes[-1][16], nodes[-1][19] = nodes[-1][19], nodes[-1][16]
                    nodes[-1][17], nodes[-1][19] = nodes[-1][19], nodes[-1][17]


    T = np.array(els)
                
    return T, np.array(mids), order

In [257]:
def load_node_sets(geometry):
    nodes_set = {}
    
    for child in geometry.iter("NodeSet"):
        name = child.attrib["name"]
                
        for nodeid in child.iter("node"):
            nid = int(nodeid.attrib["id"])-1
            nodes_set[nid] = name

            
    return nodes_set

In [258]:
def load_dirichlet(boundaries, nodes_set, dt):
    allbc = {}
    
    names = [nodes_set[k] for k in nodes_set]
    
    result = np.zeros((len(nodes_set), 4))
    result[:, 0] = np.array([k for k in node_set])
    
    for child in boundaries.iter("fix"):
        name = child.attrib["node_set"]
        
        if not name in names:
            print("Sideset {} not present, skipping".format(name))
            continue

        bc = child.attrib["bc"]
        bcs = bc.split(",")

        value = np.array([0.,0.,0.])

        if not "x" in bcs:
            value[0] = np.NAN
        if not "y" in bcs:
            value[1] = np.NAN
        if not "z" in bcs:
            value[2] = np.NAN

        for i,k in enumerate(node_set):
            if node_set[k] == name:
                result[i, 1:] = value
                                

    for child in boundaries.iter("prescribe"):
        name = child.attrib["node_set"]
        if not name in names:
            print("Sideset {} not present, skipping".format(name))
            continue

        bc = child.attrib["bc"]
        bcs = bc.split(",")
        val = float(child.find("scale").text) * dt
        value = np.array([val, val, val])
        
        if not "x" in bcs:
            value[0] = np.NAN
        if not "y" in bcs:
            value[1] = np.NAN
        if not "z" in bcs:
            value[2] = np.NAN

        for i,k in enumerate(node_set):
            if node_set[k] == name:
                result[i, 1:] = value


#             for (const tinyxml2::XMLElement *child = boundaries->FirstChildElement("vector_bc"); child != NULL; child = child->NextSiblingElement("vector_bc"))
#             {
#                 const std::string name = std::string(child->Attribute("node_set"));
#                 if (names.find(name) == names.end())
#                 {
#                     logger().error("Sideset {} not present, skipping", name);
#                     continue;
#                 }
#                 const int id = names.at(name);
#                 const std::string centers = resolve_path(std::string(child->Attribute("centers")), root_file);
#                 const std::string values = resolve_path(std::string(child->Attribute("values")), root_file);
#                 const std::string rbf = "thin_plate"; //TODO
#                 const double eps = 1e-3;              //TODO
#                 //TODO add is x,y,z

#                 Eigen::MatrixXd centers_mat, values_mat;
#                 read_matrix(centers, centers_mat);
#                 read_matrix(values, values_mat);

#                 RBFInterpolation interp(values_mat, centers_mat, rbf, eps);
#                 logger().trace("adding vector Dirichlet id={} centers={} values={} rbf={} eps={}", id, centers, values, rbf, eps);

#                 gproblem.add_dirichlet_boundary(
#                     id, [interp](double x, double y, double z, double t) {
#                         Eigen::Matrix<double, 3, 1> v;
#                         v[0] = x;
#                         v[1] = y;
#                         v[2] = z;
#                         return interp.interpolate(v);
#                     },
#                     true, true, true, get_interpolation(gproblem.is_time_dependent()));
#             }

#             const bool is_time_dept = gproblem.is_time_dependent();
#             for (const tinyxml2::XMLElement *child = boundaries->FirstChildElement("scaling"); child != NULL; child = child->NextSiblingElement("scaling"))
#             {
#                 const std::string centres = std::string(child->Attribute("center"));
#                 const std::string factors = std::string(child->Attribute("factor"));
#                 const std::string name = std::string(child->Attribute("node_set"));
#                 if (names.find(name) == names.end())
#                 {
#                     logger().error("Sideset {} not present, skipping", name);
#                     continue;
#                 }
#                 const int id = names.at(name);

#                 const auto centrec = StringUtils::split(centres, ",");
#                 if (centrec.size() != 3)
#                 {
#                     logger().error("Skipping scaling, center is not 3d");
#                     continue;
#                 }
#                 const Eigen::Vector3d center(
#                     atof(centrec[0].c_str()),
#                     atof(centrec[1].c_str()),
#                     atof(centrec[2].c_str()));

#                 const double scaling = atof(factors.c_str());
#                 logger().trace("adding scaling Dirichlet id={} center=({}) scaling={}", id, center.transpose(), scaling);
#                 gproblem.add_dirichlet_boundary(
#                     id, [center, scaling, is_time_dept](double x, double y, double z, double t) {
#                         Eigen::Matrix<double, 3, 1> v;
#                         Eigen::Matrix<double, 3, 1> target;
#                         v[0] = x;
#                         v[1] = y;
#                         v[2] = z;
#                         target = v;

#                         const double s = is_time_dept ? (scaling * t) : scaling;
#                         target -= center;
#                         target *= s;
#                         target += center;
#                         return (target - v).eval();
#                     },
#                     true, true, true);
#             }
#         }

    return result

In [259]:
def load_neumann(loads, names, dt):
    result = []
    
    if loads is None:
        return result
    
    for child in loads.iter("surface_load"):
        name = child.attrib["surface"]
        btype = child.attrib["type"]
        assert(name in names)
        
        if btype == "traction":
            traction = child.find("traction").text
            scalev = np.array([1., 1., 1.])
            
            for scale in child.iter("scale"):
                scales = scale.text
                scalev *= float(scales)


            bcs = traction.split(",")
            assert(len(bcs) == 3)

            force = np.array([float(v) for v in bcs])
            force *= scalev*dt
            
            
            print("adding Neumann id={} force=({})".format(names[name], force))

            result.append({
                "type": "neumann",
                "id": names[name],
                "value": force
                #get_interpolation(gproblem.is_time_dependent())
            })
        
        elif btype == "pressure":
            pressures = child.find("pressure").text
            
            # TODO added minus here
            pressure = -float(pressures) * dt

            print("adding Pressure id={} pressure={}".format(names[name], pressure))
                        
            result.append({
                "type": "pressure",
                "id": names[name],
                "value": pressure
                #get_interpolation(gproblem.is_time_dependent())
            })
        else:
            print("Unsupported surface load {}".format(btype))

    
    return result


In [260]:
def load_surface_selection(geometry):
    n_id = 1
    names = {}
    
    selections = []
    
    for child in geometry.iter("Surface"):
        name = child.attrib["name"]
        names[name] = n_id;
        

        # TODO  only tri3
        for nodeid in child.iter("tri3"):
            ids = nodeid.text
            tt = ids.split(",")
            assert(len(tt) == 3)
            
            selections.append([n_id] + [int(v) - 1 for v in tt])

        for nodeid in child.iter("quad4"):
            ids = nodeid.text
            tt = ids.split(",")
            assert(len(tt) == 4)

            tmp.append([int(v) - 1 for v in tt])

        n_id += 1
            
    return np.array(selections), names

In [261]:
out_folder = ""
dhat = 0.001
output = "sim.vtu"

In [262]:
output_json = {}
output_json["output"] = {
    "json": "sim.json",
    "paraview": {
            "file_name": output,
            "surface": True,
            "options": {
                "material": True,
                "body_ids": True
            },
            "vismesh_rel_area": 10000000
        }
}

In [263]:
tree = ET.parse("jobs/Model1.feb")
tree.getroot()

<Element 'febio_spec' at 0x1256a82c0>

In [264]:
# -----------------------------
# Robust FEBio (v2.x + v4.x) loader wrapper (namespace + nested Model support)
# + safer material parsing (handles missing <E>/<v> etc. in FEBio 4)
# + safer element parsing (handles FEBio 4 where <Elements> may not have attrib 'mat')
# + FIX: element connectivity parsing supports comma-separated lists like "1,9,358,..."
# + FIX: surface selection parser bug (tmp not defined) + supports comma/space separated ids
# + FIX: writing .msh with meshio when elements are NOT tet4 (e.g., hex8 / tet10 / mixed)
# -----------------------------
import xml.etree.ElementTree as ET
import json
import os
import numpy as np
import re
import meshio as mio

# tree = ET.parse("../data/test.feb")
tree = ET.parse("jobs/Model1.feb")
root = tree.getroot()

# -----------------------------
# Helpers
# -----------------------------
def strip_ns(elem):
    """Remove XML namespaces in-place so .find() works with plain tag names."""
    for e in elem.iter():
        if isinstance(e.tag, str) and "}" in e.tag:
            e.tag = e.tag.split("}", 1)[1]
    return elem

def find_first(elem, paths):
    """Try multiple .find() paths and return the first match."""
    for p in paths:
        x = elem.find(p)
        if x is not None:
            return x
    return None

def unique_tags(elem):
    return sorted({e.tag for e in elem.iter() if isinstance(e.tag, str)})

def find_text(node, tag, default=None):
    """Safe helper: returns stripped text of child <tag>, or default if missing/empty."""
    if node is None:
        return default
    child = node.find(tag)
    if child is None or child.text is None:
        return default
    txt = child.text.strip()
    return txt if txt != "" else default

def parse_int_list(text):
    """
    Parse a list of ints from FEBio text.
    Supports:
      "1 9 358 112 ..."
      "1,9,358,112,..."
      "1, 9, 358, 112, ..."
    """
    if text is None:
        return []
    s = text.strip()
    if not s:
        return []
    parts = re.split(r"[,\s]+", s)
    parts = [p for p in parts if p != ""]
    return [int(p) for p in parts]

# -----------------------------
# FIXED surface selection loader (replaces your buggy load_surface_selection)
# -----------------------------
def load_surface_selection_safe(geometry_elem):
    names = []
    selections = []

    surf_parent = find_first(geometry_elem, ["Surface", ".//Surface", "Surfaces", ".//Surfaces"])
    if surf_parent is None:
        return np.zeros((0, 4), dtype=int), names

    surf_nodes = surf_parent.findall("surface")
    if not surf_nodes:
        surf_nodes = [c for c in list(surf_parent) if isinstance(c.tag, str)]

    for sidx, sn in enumerate(surf_nodes):
        sname = sn.attrib.get("name", sn.attrib.get("id", f"surface_{sidx+1}"))
        names.append(sname)

        facet_blocks = [c for c in list(sn) if isinstance(c.tag, str)]
        if not facet_blocks:
            continue

        for blk in facet_blocks:
            for f in list(blk):
                if f.text is None:
                    continue
                ids = parse_int_list(f.text)

                if len(ids) == 3:
                    ids = ids + [-1]
                elif len(ids) != 4:
                    continue

                ids0 = [(v - 1) if v != -1 else -1 for v in ids]
                selections.append(ids0)

    if not selections:
        return np.zeros((0, 4), dtype=int), names

    return np.array(selections, dtype=int), names

# -----------------------------
# FEBio 4 element fallback parser (when your load_elements expects Elements.attrib['mat'])
# -----------------------------
def fallback_load_elements_febio4(geometry_elem):
    blocks = list(geometry_elem.iter("Elements"))
    if not blocks:
        blocks = list(geometry_elem.iter("elements"))
    if not blocks:
        raise ValueError("No <Elements> blocks found under Geometry/Mesh for fallback element parsing.")

    conn_all = []
    mids_all = []
    order = 1

    def _get_mat_id(attrib):
        for k in ("mat", "mat_id", "matid", "material", "material_id"):
            if k in attrib:
                return attrib.get(k)
        return None

    def _infer_order_from_type(t):
        if not t:
            return 1
        t = t.lower()
        if "tet10" in t or "hex20" in t or "quad8" in t:
            return 2
        if "tet20" in t:
            return 3
        return 1

    for blk in blocks:
        blk_type = blk.attrib.get("type", "")
        order = max(order, _infer_order_from_type(blk_type))
        blk_mat = _get_mat_id(blk.attrib)

        elems = blk.findall("elem")
        if not elems:
            elems = blk.findall("element")

        for el in elems:
            el_mat = _get_mat_id(el.attrib)
            mat_txt = el_mat if el_mat is not None else blk_mat
            try:
                mid = int(mat_txt) + 1 if mat_txt is not None else 1
            except ValueError:
                mid = 1

            ids = parse_int_list(el.text)
            if not ids:
                continue
            ids = [i - 1 for i in ids]  # 1-based -> 0-based

            conn_all.append(ids)
            mids_all.append(mid)

    if not conn_all:
        raise ValueError("Parsed zero elements in fallback_load_elements_febio4 (no readable <elem> connectivity found).")

    # keep as python lists; writing will group by element size
    mids = np.array(mids_all, dtype=int)
    return conn_all, mids, order

# -----------------------------
# FIX: safe mesh writer that supports hex8/tet10/mixed element sizes
# -----------------------------
def safe_save_msh(path, V, T, mids):
    """
    Writes a Gmsh .msh using meshio, but chooses correct cell type based on
    number of nodes per element. Also supports mixed element types by splitting
    into multiple cell blocks.
    """
    # T can be: np.ndarray (n,k), object array, or list[list[int]]
    if isinstance(T, np.ndarray):
        if T.dtype == object:
            elems = [list(row) for row in T.tolist()]
        else:
            elems = [list(row) for row in T.tolist()]
    else:
        elems = [list(row) for row in T]

    if len(elems) == 0:
        raise ValueError("No elements to write into mesh.")

    if not isinstance(mids, np.ndarray):
        mids = np.array(mids, dtype=int)
    if len(mids) != len(elems):
        # if mids doesn't match, fall back to ones
        mids = np.ones((len(elems),), dtype=int)

    # map (nodes per element) -> meshio cell type
    cell_type_map = {
        4: "tetra",
        10: "tetra10",
        8: "hexahedron",
        20: "hexahedron20",
        6: "wedge",
        15: "wedge15",
        5: "pyramid",
        13: "pyramid13",
        3: "triangle",
        6+0: "triangle6",  # placeholder if you later have 6-node triangles; usually 6 nodes means wedge in volume
        4+0: "quad",       # placeholder if you later have quads; usually 4 nodes means tetra in volume
    }

    # group by element size
    groups = {}
    for i, conn in enumerate(elems):
        k = len(conn)
        groups.setdefault(k, []).append(i)

    cells = []
    cell_data_phys = []
    cell_data_geom = []

    for k, idxs in sorted(groups.items(), key=lambda x: x[0]):
        if k not in cell_type_map or cell_type_map[k] in ("triangle6", "quad"):
            # be strict here to avoid writing wrong type
            raise ValueError(
                f"Unsupported element connectivity size {k}. "
                f"Update cell_type_map for your element type. "
                f"Example conn length={k}."
            )

        ctype = cell_type_map[k]
        block = np.array([elems[i] for i in idxs], dtype=int)

        cells.append((ctype, block))
        tags = np.array([int(mids[i]) for i in idxs], dtype=int)
        cell_data_phys.append(tags)
        cell_data_geom.append(tags)

    mesh = mio.Mesh(
        points=np.array(V, dtype=float),
        cells=cells,
        cell_data={
            "gmsh:physical": cell_data_phys,
            "gmsh:geometrical": cell_data_geom,
        },
    )

    # use gmsh v4 by default
    mio.write(path, mesh, file_format="gmsh")

# -----------------------------
# Start
# -----------------------------
strip_ns(root)

print("root tag:", root.tag)
print("root attrib:", root.attrib)
ver = (root.attrib.get("version") or "").strip()
print("version:", ver)

if not ver:
    raise ValueError("Missing FEBio version attribute on root element (expected root.attrib['version']).")
if not (ver.startswith("2.") or ver.startswith("4.")):
    raise ValueError(f"Unsupported FEBio version: {ver} (supports 2.x and 4.x only)")

control = find_first(root, ["Control", "./Model/Control", ".//Control"])
geometry = find_first(root, ["Geometry", "./Model/Geometry", ".//Geometry", "Mesh", "./Model/Mesh", ".//Mesh"])
boundaries = find_first(root, ["Boundary", "./Model/Boundary", ".//Boundary"])
loads = find_first(root, ["Loads", "./Model/Loads", ".//Loads"])

if control is None:
    print("Available tags (unique):", unique_tags(root))
    raise ValueError("Could not find <Control> section.")
if geometry is None:
    print("Available tags (unique):", unique_tags(root))
    raise ValueError("Could not find <Geometry> or <Mesh> section.")
if boundaries is None:
    print("Warning: <Boundary> not found. Dirichlet BCs may be empty.")
if loads is None:
    print("Warning: <Loads> not found. Neumann/pressure loads may be empty.")

# ---- Control
try:
    load_control(control, output_json)
except KeyError as e:
    if str(e).strip("'\"") == "type":
        print("Warning: <analysis> has no attribute 'type' (common in FEBio 4). Using defaults.")
        output_json.setdefault("analysis", {"type": "static"})
        output_json.setdefault("time", {"time_steps": 1, "step_size": 1.0})
    else:
        raise

dt = 1
if isinstance(output_json.get("time"), dict) and "time_steps" in output_json["time"]:
    dt = output_json["time"]["time_steps"]

# ---- Materials (fallback)
try:
    materials = load_materials(root)
except AttributeError as e:
    print("Warning: load_materials failed (likely FEBio 4 material format).")
    print("Reason:", e)
    print("Falling back to minimal material parsing (id/type + raw params).")

    materials = {}
    mats_parent = find_first(root, [
        "Material", "./Model/Material", ".//Material",
        "Materials", "./Model/Materials", ".//Materials"
    ])
    if mats_parent is None:
        print("Warning: No <Material>/<Materials> found. Using empty materials.")
    else:
        for m in mats_parent.findall(".//material"):
            mid_txt = m.attrib.get("id")
            mtype = m.attrib.get("type", "unknown")
            try:
                mid = int(mid_txt) + 1 if mid_txt is not None else None
            except ValueError:
                mid = None

            params = {}
            for child in m.iter():
                if child is m:
                    continue
                if len(list(child)) == 0 and child.text is not None and child.text.strip() != "":
                    params[child.tag] = child.text.strip()

            key = mid if mid is not None else f"material_{len(materials)+1}"
            materials[key] = {"id": key if isinstance(key, int) else None, "febio_type": mtype, "params": params}

# ---- Geometry: nodes/elements
V = load_nodes(geometry)

try:
    T, mids, order = load_elements(geometry, V.shape[0], {})
except KeyError as e:
    if str(e).strip("'\"") == "mat":
        print("Warning: <Elements> has no attribute 'mat' (common in FEBio 4). Using fallback element parsing.")
        T, mids, order = fallback_load_elements_febio4(geometry)
    else:
        raise

node_set = load_node_sets(geometry)

dirichlet = np.array([])
if boundaries is not None:
    dirichlet = load_dirichlet(boundaries, node_set, dt)

surfs, names = load_surface_selection_safe(geometry)

neumann = []
if loads is not None:
    neumann = load_neumann(loads, names, dt)

has_collisions = geometry.find(".//SurfacePair") is not None

# -----------------------------
# Output JSON assembly
# -----------------------------
if has_collisions:
    output_json["contact"] = {"enabled": True, "dhat": dhat}

# Normalize materials for output
if isinstance(materials, dict):
    if len(materials) > 1:
        output_json["materials"] = [materials[k] for k in materials]
    elif len(materials) == 1:
        output_json["materials"] = list(materials.values())[0]
    else:
        output_json["materials"] = []
else:
    output_json["materials"] = materials

output_json.setdefault("boundary_conditions", {})

output_json["geometry"] = {
    "mesh": os.path.join(out_folder, "mesh.msh"),
    "surface_selection": "" if getattr(surfs, "size", 0) <= 0 else os.path.join(out_folder, "surfaces.txt"),
}

if getattr(surfs, "size", 0) > 0:
    np.savetxt(os.path.join(out_folder, "surfaces.txt"), surfs, fmt="%d")

# ---- FIX: use safe writer instead of save_msh (because your T is NOT tet4)
safe_save_msh(os.path.join(out_folder, "mesh.msh"), V, T, mids)

output_json["boundary_conditions"].setdefault("neumann_boundary", [])
output_json["boundary_conditions"].setdefault("pressure_boundary", [])

for n in neumann:
    ntype = n.get("type")
    if ntype == "neumann":
        output_json["boundary_conditions"]["neumann_boundary"].append({"id": n["id"], "value": list(n["value"])})
    elif ntype == "pressure":
        output_json["boundary_conditions"]["pressure_boundary"].append({"id": n["id"], "value": list(n["value"])})

if isinstance(dirichlet, np.ndarray) and dirichlet.size > 0:
    output_json["boundary_conditions"]["dirichlet_boundary"] = [os.path.join(out_folder, "dirichlet.txt")]
    np.savetxt(os.path.join(out_folder, "dirichlet.txt"), dirichlet)

output_json["boundary_conditions"]["rhs"] = [0, 0, 0]

with open(os.path.join(out_folder, "sim.json"), "w") as f:
    f.write(json.dumps(output_json, indent="  "))


root tag: febio_spec
root attrib: {'version': '4.0'}
version: 4.0
Reason: 'NoneType' object has no attribute 'text'
Falling back to minimal material parsing (id/type + raw params).


In [265]:
import numpy as np


def convert_hex8_to_quads(T, c):
    
    
    """
    T: (n, 8) connectivity
    returns:
      F: (n*6, 4) quads (each row is 4 node indices)
      C: (n*6,) repeated cell color/material id
    """
    T = np.asarray(T, dtype=int)
    c = np.asarray(c)

    if T.ndim != 2 or T.shape[1] != 8:
        raise ValueError(f"This function only supports hex8. shape={T.shape}")
    
    # Hex8 faces (standard ordering)
    faces = np.array([
        [0, 1, 2, 3],
        [4, 5, 6, 7],
        [0, 1, 5, 4],
        [1, 2, 6, 5],
        [2, 3, 7, 6],
        [3, 0, 4, 7],
    ], dtype=int)

    F = np.empty((T.shape[0]*6, 4), dtype=int)
    C = np.empty((T.shape[0]*6,), dtype=c.dtype)

    for i in range(T.shape[0]):
        F[i*6:(i+1)*6] = T[i, faces]
        C[i*6:(i+1)*6] = c[i]

    return F, C


In [266]:
# If T is hex8:
F, C = convert_hex8_to_quads(T, mids)

# meshplot usually uses (V, F) for face meshes
mp.plot(V, F, C, shading={"point_size": 1, "wireframe": True})


Invalid color array given! Supported are numpy arrays. <class 'numpy.ndarray'>


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0, 0.0,…

In [267]:
mio.Mesh?

Init signature:
mio.Mesh(
    points: 'ArrayLike',
    cells: 'dict[str, ArrayLike] | list[tuple[str, ArrayLike] | CellBlock]',
    point_data: 'dict[str, ArrayLike] | None' = None,
    cell_data: 'dict[str, list[ArrayLike]] | None' = None,
    field_data=None,
    point_sets: 'dict[str, ArrayLike] | None' = None,
    cell_sets: 'dict[str, list[ArrayLike]] | None' = None,
    gmsh_periodic=None,
    info=None,
)
Docstring:      <no docstring>
File:           /opt/anaconda3/lib/python3.11/site-packages/meshio/_mesh.py
Type:           type
Subclasses:     

In [268]:
root.find("Globals")

<Element 'Globals' at 0x126f5c3b0>